In [1]:
import pickle
with open("embeddings_another.pkl", "rb") as f:
    embeddings_another = pickle.load(f)
    
with open("embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

/home/user/miniconda/envs/minus_face/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embeddings.keys()

dict_keys(['student', 'teacher', 'buffalo_l', 'filenames'])

In [3]:
embeddings["student"].shape

torch.Size([79807, 512])

In [4]:
X = embeddings["teacher"]
Y = embeddings["buffalo_l"][:,0]

In [5]:
X.shape, Y.shape

(torch.Size([79807, 512]), torch.Size([79807, 512]))

In [6]:
import torch
def cosine_similarity_loss(x, y):
    sim = torch.nn.functional.cosine_similarity(x, y, dim=-1)
    loss = 1 - sim.mean()
    return loss

In [7]:
mlp = torch.nn.Sequential(
    torch.nn.Linear(512, 1024),
    torch.nn.GELU(), 
    torch.nn.Linear(1024, 1024),
    torch.nn.GELU(),
    torch.nn.Linear(1024, 1024),
    torch.nn.GELU(),
    torch.nn.Linear(1024, 512),
)

In [8]:
X.shape

torch.Size([79807, 512])

In [9]:
train_dataset = torch.utils.data.TensorDataset(X[:79000], Y[:79000])
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

val_dataset = torch.utils.data.TensorDataset(X[79000:], Y[79000:])
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlp.to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
for epoch in range(20): 
    for batch_x, batch_y in train_dataloader:
        optimizer.zero_grad()
        pred_y = mlp(batch_x.to(device))
        loss = cosine_similarity_loss(pred_y, batch_y.to(device))
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}: Train Loss = {loss.item()}")
    with torch.no_grad():
        val_losses = []
        for batch_x, batch_y in val_dataloader:
            pred_y = mlp(batch_x.to(device))
            loss = cosine_similarity_loss(pred_y, batch_y.to(device))
            val_losses.append(loss.item())
        avg_val_loss = sum(val_losses) / len(val_losses)
        print(f"Epoch {epoch}: Val Loss = {avg_val_loss}, LR = {scheduler.get_last_lr()[0]}")
    scheduler.step()

Epoch 0: Train Loss = 0.4750251770019531
Epoch 0: Val Loss = 0.454090416431427, LR = 0.001
Epoch 1: Train Loss = 0.4340209364891052
Epoch 1: Val Loss = 0.4259234987772428, LR = 0.0009938441702975688
Epoch 2: Train Loss = 0.4404214024543762
Epoch 2: Val Loss = 0.4125209129773654, LR = 0.0009755282581475768
Epoch 3: Train Loss = 0.3745730519294739
Epoch 3: Val Loss = 0.40174029423640323, LR = 0.0009455032620941839
Epoch 4: Train Loss = 0.41139495372772217
Epoch 4: Val Loss = 0.39451956290465134, LR = 0.0009045084971874737
Epoch 5: Train Loss = 0.38654661178588867
Epoch 5: Val Loss = 0.38733343894665057, LR = 0.0008535533905932737
Epoch 6: Train Loss = 0.3792259693145752
Epoch 6: Val Loss = 0.3826611042022705, LR = 0.0007938926261462366
Epoch 7: Train Loss = 0.4056485891342163
Epoch 7: Val Loss = 0.3796446415094229, LR = 0.0007269952498697733


KeyboardInterrupt: 

In [ ]:
# teacher has better features so it can fit better, but will it work? different image? We should use teacher because this is not about finding but reverting

In [ ]:
embeddings.keys()

dict_keys(['student', 'teacher', 'buffalo_l', 'filenames'])

In [24]:
idx = 7
out = mlp(embeddings["student"][idx:idx+1].to(device))
loss = cosine_similarity_loss(out, embeddings["buffalo_l"][idx:idx+1].to(device))
print("Test Loss:", loss.item())
torch.save(out.cpu(), "student_to_buffalo_l.pt") 

Test Loss: 0.47555238008499146


In [25]:
embeddings["filenames"][idx]

'training/032816.jpg'

In [ ]:
import sys 
from minusface import MinusBackbone
model = MinusBackbone(mode='stage1') 

In [ ]:
import torch
ckpt_path = "../../../../minusface_stage1.pth"
model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))

<All keys matched successfully>

In [ ]:
def preprocess_image(image_path):
    from PIL import Image
    from torchvision import transforms

    input_image = Image.open(image_path).convert('RGB')
    preprocess = transforms.Compose([
        transforms.Resize((112, 112)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])
    input_tensor = preprocess(input_image)
    input_batch = input_tensor.unsqueeze(0)  # Create a mini-batch as expected by the model
    return input_batch

In [ ]:
model = model.eval()
model = model.to('cuda')

In [ ]:
from PIL import Image
res = model(preprocess_image(embeddings["filenames"][0].replace("0018", "0018")).to('cuda'))[5][0].permute(1, 2, 0).cpu().detach().numpy()
res = (res - res.min()) / (res.max() - res.min())
Image.fromarray((res * 255).astype('uint8')).save("debug.png")